# `vad_funasr` with VoiceHub

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadirnar/voicehub/blob/main/notebooks/models/vad_funasr.ipynb)

- Task: **Voice activity detection**
- Hugging Face ID: [`funasr/fsmn-vad`](https://huggingface.co/funasr/fsmn-vad)

Install VoiceHub using the [installation guide](https://kadirnar.github.io/voicehub/getting-started/installation/)
before opening this model workflow. This notebook contains no package-install cell.

The registry check is safe to run without downloading weights. Inference is disabled by default.


In [ ]:
from pathlib import Path

RUN_INFERENCE = False
MODEL_TYPE = 'vad_funasr'
CHECKPOINT = 'funasr/fsmn-vad'
DEVICE = "cpu"
AUDIO_FILE = Path("speech.wav")


## Inspect registry support


In [ ]:
from voicehub import get_model_spec

model_spec = get_model_spec(MODEL_TYPE)
assert model_spec.task.value == 'voice-activity-detection'
assert model_spec.default_model_path == CHECKPOINT
print("task:", model_spec.task.value)
print("checkpoint:", model_spec.default_model_path)
print("capabilities:", ", ".join(model_spec.capabilities))
print("training:", model_spec.training.support.value)


## Run inference

Uses the FSMN endpoint model with speech padding and maximum-segment limits.

Endpoint behavior is model-specific; calibrate padding before composing it with ASR chunks.

Place an authorized recording at `speech.wav`, then set `RUN_INFERENCE = True`.


In [ ]:
if RUN_INFERENCE:
    from voicehub import AutoModelForVoiceActivityDetection

    if not AUDIO_FILE.is_file():
        raise FileNotFoundError(AUDIO_FILE)
    model = AutoModelForVoiceActivityDetection.from_pretrained(
        CHECKPOINT,
        model_type=MODEL_TYPE,
        device=DEVICE,
        lazy_load=True,
    )
    output = model.detect(
        AUDIO_FILE,
        threshold=0.5,
        speech_pad_ms=200,
        max_speech_duration_s=30.0,
    )
    for segment in output.segments:
        print(segment.start, segment.end, segment.score)


## Next

See the [inference guide](https://kadirnar.github.io/voicehub/guides/inference/) and [model catalog](https://kadirnar.github.io/voicehub/models/) for the shared runtime contract and model-specific limitations.
